In [ ]:
# Import required libraries
import json
from bs4 import BeautifulSoup
import os
import re

# Set file paths
html_path = "../realCase/EllieTale-WebXR-main/html/Beach.html"  # Replace with your HTML file path
priority_json_path = "./priority.json"  # Replace with priority JSON file path
output_path = "../realCase/EllieTale-WebXR-main/html/priority_updated_scene.html"  # Replace with output file path

In [6]:
# Read priority data
with open(priority_json_path, 'r') as f:
    priority_data = json.load(f)

# Extract priority lists with exclusive sets
level1 = set(priority_data.get('visibleInFrustumObjectsSource', []))
level2 = set(priority_data.get('visibleObjectsSource', [])) - level1
level3 = set(priority_data.get('allObjectsSource', [])) - set(priority_data.get('visibleObjectsSource', []))
print(level1)
print(level2)
print(level3)

{'item/Beach_Seashell_lit_v1.02.glb', 'item/Beach_Environment_v1.02.glb'}
{'item/Beach_prop_fishingRod_01.glb', 'item/Beach_prop_ship_01.glb', 'item/Elly_2k_03.glb', 'item/Beach_prop_whale_01.glb', 'item/Beach_prop_ball_01.glb', 'item/Beach_prop_sharky_01.glb', 'item/Beach_prop_octopus_01.glb', 'item/Beach_prop_shrimp_01.glb'}
{'item/Beach_Seashell_particles_half_v1.01.glb', 'item/Noah_hunter_v1.01.glb'}


In [7]:
# Read HTML file
with open(html_path, 'r') as f:
    html_content = f.read()

# Parse HTML
soup = BeautifulSoup(html_content, 'html.parser')

# Initialize statistics
stats = {
    'total_resources': 0,
    'level1': 0,
    'level2': 0,
    'level3': 0,
    'unclassified_models': 0  # 添加未分类模型计数器
}

# Process all <a-asset-item> elements
for item in soup.find_all('a-asset-item'):
    stats['total_resources'] += 1
    src = item.get('src', '')
    
    if not src:
        continue
    
    # 检查是否为模型文件
    is_model = src.lower().endswith(('.glb', '.gltf'))
    
    # Determine priority level
    priority = None
    if src in level1:
        priority = '1'
        stats['level1'] += 1
    elif src in level2:
        priority = '2'
        stats['level2'] += 1
    elif src in level3:
        priority = '3'
        stats['level3'] += 1
    else:
        # 如果是模型文件但未在优先级列表中找到
        if is_model:
            print(f"ERROR: Model file not found in priority lists: {src}")
            stats['unclassified_models'] += 1
    
    # Update priority attribute
    if priority:
        item['priority'] = priority

# Save modified HTML
with open(output_path, 'w') as f:
    f.write(str(soup))

# Print statistics
print("="*50)
print("Resource priority update completed")
print("="*50)
print(f"Total resources processed: {stats['total_resources']}")
print(f"Level 1 resources (visible in frustum): {stats['level1']}")
print(f"Level 2 resources (visible but not in frustum): {stats['level2']}")
print(f"Level 3 resources (not visible): {stats['level3']}")
print(f"Unclassified models: {stats['unclassified_models']}")
print(f"Other resources: {stats['total_resources'] - stats['level1'] - stats['level2'] - stats['level3'] - stats['unclassified_models']}")
print(f"Output file saved to: {output_path}")
print("="*50)

Resource priority update completed
Total resources processed: 32
Level 1 resources (visible in frustum): 2
Level 2 resources (visible but not in frustum): 8
Level 3 resources (not visible): 2
Unclassified models: 0
Other resources: 20
Output file saved to: ../realCase/EllieTale-WebXR-main/html/priority_updated_scene.html
